# YZM304 – II. Proje: CNN ile Özellik Çıkarma ve Sınıflandırma
**Veri Seti:** CIFAR-10 | **Modeller:** LeNet-5, LeNet-5+BN+Dropout, ResNet-18, SVM Hibrit

> GPU'yu aktif etmek için: **Çalışma zamanı → Çalışma zamanı türünü değiştir → T4 GPU**

In [ ]:
# Gerekli kütüphaneleri kur
!pip install -q seaborn scikit-learn torch torchvision

In [ ]:
import os, numpy as np, torch, torch.nn as nn, torch.nn.functional as F
import torchvision, torchvision.transforms as transforms, torchvision.models as tv_models
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (confusion_matrix, classification_report,
                              accuracy_score)

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Cihaz: {DEVICE}')
os.makedirs('results', exist_ok=True)

CIFAR10_CLASSES = ['airplane','automobile','bird','cat','deer',
                   'dog','frog','horse','ship','truck']

## 1. Veri Yükleme – CIFAR-10

In [ ]:
BATCH_SIZE = 64
_MEAN = (0.4914, 0.4822, 0.4465)
_STD  = (0.2470, 0.2435, 0.2616)

train_tf = transforms.Compose([
    transforms.RandomHorizontalFlip(),
    transforms.RandomCrop(32, padding=4),
    transforms.ToTensor(),
    transforms.Normalize(_MEAN, _STD),
])
test_tf = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(_MEAN, _STD),
])

train_set = torchvision.datasets.CIFAR10(root='./data', train=True,  download=True, transform=train_tf)
test_set  = torchvision.datasets.CIFAR10(root='./data', train=False, download=True, transform=test_tf)
train_loader = DataLoader(train_set, batch_size=BATCH_SIZE, shuffle=True,  num_workers=2, pin_memory=True)
test_loader  = DataLoader(test_set,  batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

print(f'Eğitim: {len(train_set)} örnek | Test: {len(test_set)} örnek | Boyut: 3x32x32')

## 2. Model Tanımları

In [ ]:
class LeNet5(nn.Module):
    """Model 1: LeNet-5 tarzı temel CNN (3x32x32 giriş, 10 sınıf)"""
    def __init__(self, num_classes=10):
        super().__init__()
        self.conv1 = nn.Conv2d(3, 6, kernel_size=5)
        self.pool  = nn.MaxPool2d(2, 2)
        self.conv2 = nn.Conv2d(6, 16, kernel_size=5)
        self.fc1   = nn.Linear(16*5*5, 120)
        self.fc2   = nn.Linear(120, 84)
        self.fc3   = nn.Linear(84, num_classes)

    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))
        x = self.pool(F.relu(self.conv2(x)))
        x = x.view(x.size(0), -1)
        return self.fc3(F.relu(self.fc2(F.relu(self.fc1(x)))))


class LeNet5Improved(nn.Module):
    """Model 2: LeNet-5 + BatchNorm + Dropout"""
    def __init__(self, num_classes=10):
        super().__init__()
        self.conv1   = nn.Conv2d(3, 6, kernel_size=5)
        self.bn1     = nn.BatchNorm2d(6)
        self.pool    = nn.MaxPool2d(2, 2)
        self.conv2   = nn.Conv2d(6, 16, kernel_size=5)
        self.bn2     = nn.BatchNorm2d(16)
        self.dropout = nn.Dropout(0.5)
        self.fc1     = nn.Linear(16*5*5, 120)
        self.fc2     = nn.Linear(120, 84)
        self.fc3     = nn.Linear(84, num_classes)

    def forward(self, x):
        x = self.pool(F.relu(self.bn1(self.conv1(x))))
        x = self.pool(F.relu(self.bn2(self.conv2(x))))
        x = x.view(x.size(0), -1)
        x = self.dropout(F.relu(self.fc1(x)))
        x = self.dropout(F.relu(self.fc2(x)))
        return self.fc3(x)

## 3. Yardımcı Fonksiyonlar (Eğitim & Değerlendirme)

In [ ]:
def train_epoch(model, loader, criterion, optimizer):
    model.train()
    loss_sum, correct, total = 0.0, 0, 0
    for X, y in loader:
        X, y = X.to(DEVICE), y.to(DEVICE)
        optimizer.zero_grad()
        out = model(X)
        loss = criterion(out, y)
        loss.backward(); optimizer.step()
        loss_sum += loss.item()
        correct  += out.argmax(1).eq(y).sum().item()
        total    += y.size(0)
    return loss_sum/len(loader), 100*correct/total

def eval_epoch(model, loader, criterion):
    model.eval()
    loss_sum, correct, total = 0.0, 0, 0
    with torch.no_grad():
        for X, y in loader:
            X, y = X.to(DEVICE), y.to(DEVICE)
            out  = model(X)
            loss_sum += criterion(out, y).item()
            correct  += out.argmax(1).eq(y).sum().item()
            total    += y.size(0)
    return loss_sum/len(loader), 100*correct/total

def fit(model, epochs, lr, name):
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    model.to(DEVICE)
    hist = {'tl':[],'vl':[],'ta':[],'va':[]}
    for ep in range(epochs):
        tl, ta = train_epoch(model, train_loader, criterion, optimizer)
        vl, va = eval_epoch(model,  test_loader,  criterion)
        for k,v in zip(hist, [tl,vl,ta,va]): hist[k].append(v)
        if (ep+1)%5==0 or ep==0:
            print(f'  [{ep+1:3d}/{epochs}] train loss={tl:.4f} acc={ta:.1f}%  |  test loss={vl:.4f} acc={va:.1f}%')
    print(f'  >> {name} final test accuracy: {hist["va"][-1]:.2f}%')
    return hist

def get_preds(model, loader):
    model.eval()
    preds, labels = [], []
    with torch.no_grad():
        for X, y in loader:
            preds.extend(model(X.to(DEVICE)).argmax(1).cpu().numpy())
            labels.extend(y.numpy())
    return np.array(labels), np.array(preds)

def show_curves(hist, name):
    fig, (a1, a2) = plt.subplots(1, 2, figsize=(13, 4))
    ep = range(1, len(hist['tl'])+1)
    a1.plot(ep, hist['tl'], label='Train'); a1.plot(ep, hist['vl'], label='Test')
    a1.set_title(f'{name} – Loss'); a1.set_xlabel('Epoch'); a1.legend(); a1.grid(True)
    a2.plot(ep, hist['ta'], label='Train'); a2.plot(ep, hist['va'], label='Test')
    a2.set_title(f'{name} – Accuracy (%)'); a2.set_xlabel('Epoch'); a2.legend(); a2.grid(True)
    plt.tight_layout(); plt.savefig(f'results/{name}_curves.png', dpi=150); plt.show()

def show_cm(y_true, y_pred, name):
    cm = confusion_matrix(y_true, y_pred)
    plt.figure(figsize=(9,7))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=CIFAR10_CLASSES, yticklabels=CIFAR10_CLASSES)
    plt.title(f'{name} – Confusion Matrix')
    plt.ylabel('Gerçek'); plt.xlabel('Tahmin')
    plt.tight_layout(); plt.savefig(f'results/{name}_cm.png', dpi=150); plt.show()
    return cm

## Model 1 – LeNet-5 Temel CNN

In [ ]:
print('='*60)
print('MODEL 1: LeNet-5 Temel CNN')
print('='*60)
model1 = LeNet5()
print(f'Parametre sayısı: {sum(p.numel() for p in model1.parameters()):,}')
hist1 = fit(model1, epochs=20, lr=0.001, name='Model1_LeNet5')
y1, p1 = get_preds(model1, test_loader)
print(classification_report(y1, p1, target_names=CIFAR10_CLASSES))
show_curves(hist1, 'Model1_LeNet5')
show_cm(y1, p1, 'Model1_LeNet5')
torch.save(model1.state_dict(), 'results/model1.pth')

## Model 2 – LeNet-5 + BatchNorm + Dropout

In [ ]:
print('='*60)
print('MODEL 2: LeNet-5 + BatchNorm + Dropout')
print('='*60)
model2 = LeNet5Improved()
print(f'Parametre sayısı: {sum(p.numel() for p in model2.parameters()):,}')
hist2 = fit(model2, epochs=20, lr=0.001, name='Model2_Improved')
y2, p2 = get_preds(model2, test_loader)
print(classification_report(y2, p2, target_names=CIFAR10_CLASSES))
show_curves(hist2, 'Model2_Improved')
show_cm(y2, p2, 'Model2_Improved')
torch.save(model2.state_dict(), 'results/model2.pth')

## Model 3 – ResNet-18 (Pretrained, Fine-Tuning)

In [ ]:
print('='*60)
print('MODEL 3: ResNet-18 Pretrained + Fine-Tuning')
print('='*60)
try:
    model3 = tv_models.resnet18(weights=tv_models.ResNet18_Weights.IMAGENET1K_V1)
except AttributeError:
    model3 = tv_models.resnet18(pretrained=True)

model3.conv1   = nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1, bias=False)
model3.maxpool = nn.Identity()
model3.fc      = nn.Linear(model3.fc.in_features, 10)
print(f'Parametre sayısı: {sum(p.numel() for p in model3.parameters()):,}')

hist3 = fit(model3, epochs=10, lr=0.0001, name='Model3_ResNet18')
y3, p3 = get_preds(model3, test_loader)
print(classification_report(y3, p3, target_names=CIFAR10_CLASSES))
show_curves(hist3, 'Model3_ResNet18')
show_cm(y3, p3, 'Model3_ResNet18')
torch.save(model3.state_dict(), 'results/model3.pth')

## Model 4 – Hibrit: ResNet-18 Özellik Çıkarımı + SVM

In [ ]:
print('='*60)
print('MODEL 4: Hibrit – CNN Özellik Çıkarımı + SVM')
print('='*60)

def extract_features(model, loader):
    extractor = nn.Sequential(*list(model.children())[:-1]).eval().to(DEVICE)
    feats, labs = [], []
    with torch.no_grad():
        for X, y in loader:
            f = extractor(X.to(DEVICE)).view(X.size(0), -1)
            feats.append(f.cpu().numpy()); labs.append(y.numpy())
    return np.vstack(feats), np.concatenate(labs)

print('Özellik çıkarımı yapılıyor...')
X_tr, y_tr = extract_features(model3, train_loader)
X_te, y_te = extract_features(model3, test_loader)

print(f'\nÖzellik Seti Boyutları:')
print(f'  X_train : {X_tr.shape}  (örnek x özellik_boyutu)')
print(f'  y_train : {y_tr.shape}')
print(f'  X_test  : {X_te.shape}')
print(f'  y_test  : {y_te.shape}')
print(f'  Özellik boyutu : {X_tr.shape[1]}')
print(f'  Eğitim örnek   : {X_tr.shape[0]}')
print(f'  Test örnek     : {X_te.shape[0]}')

np.save('results/X_train_features.npy', X_tr)
np.save('results/y_train_labels.npy',   y_tr)
np.save('results/X_test_features.npy',  X_te)
np.save('results/y_test_labels.npy',    y_te)
print('\n.npy dosyaları results/ klasörüne kaydedildi.')

scaler    = StandardScaler()
X_tr_sc   = scaler.fit_transform(X_tr)
X_te_sc   = scaler.transform(X_te)

print('\nSVM eğitiliyor (RBF, C=10)...')
svm = SVC(kernel='rbf', C=10.0, gamma='scale', random_state=42)
svm.fit(X_tr_sc, y_tr)
svm_preds = svm.predict(X_te_sc)
svm_acc   = accuracy_score(y_te, svm_preds) * 100
print(f'\nModel 4 (SVM) Test Doğruluğu: {svm_acc:.2f}%')
print(classification_report(y_te, svm_preds, target_names=CIFAR10_CLASSES))
show_cm(y_te, svm_preds, 'Model4_SVM')

## Genel Sonuç Tablosu

In [ ]:
print('='*60)
print('GENEL SONUÇ TABLOSU')
print('='*60)
results = [
    ('Model 1  LeNet-5 (temel)',             accuracy_score(y1, p1)*100),
    ('Model 2  LeNet-5 + BN + Dropout',      accuracy_score(y2, p2)*100),
    ('Model 3  ResNet-18 (pretrained)',       accuracy_score(y3, p3)*100),
    ('Model 4  SVM + ResNet-18 özellikleri', svm_acc),
    ('Model 5  ResNet-18 CNN (= Model 3)',    accuracy_score(y3, p3)*100),
]
print(f'{"Model":<42} {"Test Doğruluğu":>14}')
print('-'*58)
for name, score in results:
    print(f'{name:<42} {score:>13.2f}%')

with open('results/results_summary.csv','w') as f:
    f.write('Model,Test_Accuracy\n')
    for n,s in results: f.write(f'{n},{s:.2f}\n')
print('\nresults/results_summary.csv kaydedildi.')

## Sonuçları GitHub'a Kaydet (İsteğe Bağlı)
Aşağıdaki hücreyi çalıştırmak için GitHub token gereklidir.

In [ ]:
# Bu hücreyi çalıştırmak zorunda değilsin.
# Sonuçları indirmek için: Sol panel > Dosyalar > results/ klasörünü indir
print('Sonuçları indirmek için:')
print('Sol panel (📁) > results/ klasörü > sağ tık > İndir')